# 23 评测（Evaluation） / Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaiYuBY/use_token_to_learn/blob/main/docs/23-evaluation/23-evaluation.ipynb)

[📖 讲解 / Notes](https://baiyuby.github.io/use_token_to_learn/23-evaluation/README.html) · [🎮 交互演示 / Playground](https://baiyuby.github.io/use_token_to_learn/23-evaluation/interactive/evaluation.html) · [🧑‍💻 GitHub](https://github.com/BaiYuBY/use_token_to_learn/tree/main/docs/23-evaluation)

这个 notebook 由 `scripts/make_notebooks.py` 根据本章 `code/` 目录下的脚本自动生成，代码与脚本完全一致：每个函数或类是一个单元格，
可以逐个修改、重新运行。最后一个单元格运行本章的演示（带 assert 检查）。

Generated by `scripts/make_notebooks.py` from this chapter's `code/` scripts, with identical code: one cell per function or
class, so you can edit and re-run them one at a time. The last cell runs the chapter's demo (with its assert checks).

In [ ]:
# 环境 / setup: find the repository, or clone it (e.g. in Colab); some chapters import code from earlier ones
# 找到仓库，找不到就克隆一份（例如在 Colab 中）；有些章节会导入前面章节的代码
import subprocess
import sys
from pathlib import Path

CHAPTER = "23-evaluation"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / CHAPTER).is_dir()), None)
if ROOT is None:
    ROOT = Path.cwd() / "use_token_to_learn"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/BaiYuBY/use_token_to_learn.git", str(ROOT)], check=True)
CODE = ROOT / "docs" / CHAPTER / "code"
sys.path.insert(0, str(CODE))
print("code:", CODE)

## `benchmarks.py`

```text
Benchmarks are samples: noise, paired comparisons and leaderboards / 基准是一次抽样：噪声、配对比较与排行榜.

Run / 运行:
    python docs/23-evaluation/code/benchmarks.py

Covers / 包含:
    1. a benchmark score is an estimate: its standard error is sqrt(p(1-p)/n)
       基准分数是一个估计值：标准误差是 sqrt(p(1-p)/n)
    2. item response theory: models have an ability, questions a difficulty; comparing two models on the
       same questions (paired) needs far fewer questions than comparing two separate scores
       项目反应理论：模型有能力值，题目有难度；在同一批题上做配对比较，比比较两个独立分数需要的题目少得多
    3. leaderboards: how often the ranking (and the top spot) is right, as the benchmark grows; an easy
       (saturated) benchmark separates models worse
       排行榜：随着题目增多，排名（以及第一名）有多大概率是对的；太容易（饱和）的基准区分度更差
    4. averages hide profiles: per-category scores (the radar chart) can reverse a ranking
       平均分会掩盖能力结构：分类别的分数（雷达图）可能让排名反过来

Simulation only (numpy), a few seconds. 纯模拟（numpy），几秒钟。
```

In [ ]:
__file__ = str(CODE / "benchmarks.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import numpy as np

In [ ]:
RNG = np.random.default_rng(0)

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

### A population of questions and models, in item response theory / 项目反应理论下的题目总体与模型

In [ ]:
def make_items(n, shift=0.0, rng=RNG):
    """Question difficulties; shift < 0 makes an easier benchmark. 题目难度；shift < 0 表示更容易的基准。"""
    return rng.normal(shift, 1.5, n)

In [ ]:
def answer(abilities, difficulties, rng=RNG):
    """1 if model m gets item i right: P = sigmoid(ability - difficulty) (the Rasch model).
    模型 m 答对第 i 题的概率 = sigmoid(能力 - 难度)（Rasch 模型）。"""
    p = sigmoid(abilities[:, None] - difficulties[None, :])
    return (rng.random(p.shape) < p).astype(float)

In [ ]:
def true_accuracy(ability, population):
    return sigmoid(ability - population).mean()

In [ ]:
def kendall_tau(a, b):
    n, s = len(a), 0
    for i in range(n):
        for j in range(i + 1, n):
            s += np.sign(a[i] - a[j]) * np.sign(b[i] - b[j])
    return s / (n * (n - 1) / 2)

In [ ]:
def paired_significant(x, y, n_boot=400, rng=RNG):
    """Paired bootstrap: resample the same questions for both models. 配对自助法：两个模型用同一批重抽样的题。"""
    idx = rng.integers(0, len(x), (n_boot, len(x)))
    diffs = x[idx].mean(1) - y[idx].mean(1)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return lo > 0 or hi < 0

In [ ]:
def unpaired_significant(x, y):
    """Treat the two scores as independent: difference > 1.96 standard errors. 把两个分数当作独立的。"""
    p1, p2, n = x.mean(), y.mean(), len(x)
    se = np.sqrt(p1 * (1 - p1) / n + p2 * (1 - p2) / n)
    return abs(p1 - p2) > 1.96 * se

In [ ]:
def leaderboard_stats(abilities, n_items, shift=0.0, trials=300, rng=RNG):
    """Average Kendall tau with the true ranking, and how often the true best model is ranked first.
    与真实排名的平均 Kendall tau，以及真正最好的模型排第一的比例。"""
    taus, top = [], 0
    for _ in range(trials):
        scores = answer(abilities, make_items(n_items, shift, rng), rng).mean(1)
        scores = scores + rng.random(len(scores)) * 1e-9  # break ties at random / 随机打破平局
        taus.append(kendall_tau(scores, abilities))
        top += scores.argmax() == abilities.argmax()
    return float(np.mean(taus)), top / trials

In [ ]:
def main():
    population = make_items(20000)

    print("== 1. a score is an estimate / 分数是一个估计值 ==")
    for n in [100, 1000, 10000]:
        se = np.sqrt(0.7 * 0.3 / n)
        draws = (RNG.random((2000, n)) < 0.7).mean(1)
        print(f"  true accuracy 70%, {n:5d} questions: 95% interval ±{1.96 * se:.1%} (simulated spread ±{1.96 * draws.std():.1%})")
        assert abs(draws.std() - se) < 0.15 * se

    print("\n== 2. paired vs unpaired comparison / 配对与不配对的比较 ==")
    a, b = np.array([1.0]), np.array([0.75])
    print(f"  model A {true_accuracy(a[0], population):.1%} vs model B {true_accuracy(b[0], population):.1%} on the whole question population")
    power = {}
    for n in [200, 500, 1000, 2000]:
        hits_p = hits_u = 0
        for _ in range(150):
            items = make_items(n)
            x, y = answer(a, items)[0], answer(b, items)[0]
            hits_p += paired_significant(x, y)
            hits_u += unpaired_significant(x, y)
        power[n] = (hits_u / 150, hits_p / 150)
        print(f"  {n:4d} shared questions: difference detected {hits_u / 150:4.0%} of the time unpaired, {hits_p / 150:4.0%} paired")
    assert all(p > u for u, p in power.values()) and sum(p - u for u, p in power.values()) > 0.2  # pairing removes question-difficulty noise / 配对去掉了题目难度带来的噪声

    print("\n== 3. leaderboards / 排行榜 ==")
    abilities = np.linspace(0.3, 1.1, 10)  # 10 models, about 1.5 points apart / 10 个模型，相邻约差 1.5 分
    accs = [true_accuracy(t, population) for t in abilities]
    print(f"  10 models with true accuracy {accs[0]:.0%} ... {accs[-1]:.0%}, neighbours about {np.mean(np.diff(accs)):.1%} apart")
    table = {}
    for n in [100, 500, 2000]:
        table[n] = leaderboard_stats(abilities, n)
        print(f"  {n:4d} questions: Kendall tau with the true ranking {table[n][0]:.2f}; true best ranked first {table[n][1]:.0%}")
    easy = leaderboard_stats(abilities, 500, shift=-3.0)
    easy_accs = [sigmoid(t - make_items(20000, -3.0)).mean() for t in abilities[[0, -1]]]
    print(f"  an easy (saturated) benchmark, 500 questions: scores {easy_accs[0]:.0%} ... {easy_accs[1]:.0%}; tau {easy[0]:.2f}, best first {easy[1]:.0%}")
    assert table[100][0] < table[500][0] < table[2000][0]
    assert table[100][1] < 0.6 < table[2000][1]
    assert easy[0] < table[500][0] - 0.1

    print("\n== 4. averages hide profiles / 平均分掩盖能力结构 ==")
    cats = ["knowledge", "reasoning", "math", "code", "instructions", "safety"]
    profiles = {"model A": [0.9, 0.6, 0.5, 0.55, 0.85, 0.9], "model B": [0.6, 0.85, 0.9, 0.85, 0.65, 0.45]}
    for name, p in profiles.items():
        print(f"  {name}: " + ", ".join(f"{c} {v:.0%}" for c, v in zip(cats, p)) + f" | average {np.mean(p):.1%}")
    uses = {"a coding assistant": [0, 0.3, 0.3, 0.4, 0, 0], "a customer-service bot": [0.4, 0, 0, 0, 0.3, 0.3]}  # what each use weighs / 各用途看重的类别
    for use, w in uses.items():
        print(f"  weighted for {use}: " + ", ".join(f"{name} {np.dot(w, p):.1%}" for name, p in profiles.items()))
    assert abs(np.mean(profiles["model A"]) - np.mean(profiles["model B"])) < 0.01
    print("  same average, very different models: report the categories, and weight them for your use case")

### 运行演示 / Run the demo (`benchmarks.py`)

In [ ]:
main()

## `contamination.py`

```text
Benchmark contamination: measuring it and detecting it / 基准污染：测量与检测.

Run / 运行:
    python docs/23-evaluation/code/contamination.py

Covers / 包含:
    1. a benchmark: 200 two-digit multiplication problems, plus a "fresh" twin set of 200 more from the
       same distribution that no model ever sees
       一个基准：200 道两位数乘法题，以及从同一分布另取的 200 道“新题”，任何模型都没见过
    2. contamination: the same tiny model trained with the benchmark leaked into its data 0, 1 or 4 times;
       its benchmark score rises, its score on fresh problems does not
       污染：同一个小模型，训练数据里混入基准 0、1、4 次；基准分数上涨，新题分数不变
    3. detection: exact-match search of the training data (fooled by a rewritten leak), and the model's own
       loss on benchmark items vs fresh items (a membership-inference test)
       检测：在训练数据里精确匹配（换个写法就漏掉），以及比较模型在基准题和新题上的损失（成员推断）

A tiny GPT (chapter 6's) on character-level arithmetic, about a minute on a laptop CPU.
用 06 章的小 GPT 做字符级算术，笔记本 CPU 上约一分钟。
```

In [ ]:
__file__ = str(CODE / "contamination.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import random
import sys
from pathlib import Path
import torch
import torch.nn.functional as F

In [ ]:
sys.path.insert(0, str(Path(__file__).resolve().parent.parent.parent / "06-pretraining" / "code"))

In [ ]:
from pretrain import GPT  # noqa: E402

In [ ]:
CHARS = "0123456789*=;"
STOI = {c: i for i, c in enumerate(CHARS)}
Q_LEN, A_LEN = 6, 4  # "37*58=" then the answer reversed and padded to 4 digits / 答案倒序、补足 4 位

In [ ]:
def fmt(a, b):
    """'37*58=' + '6412' (2146 reversed; reversing helps a small model, which writes the lowest digit first) + ';'
    答案倒着写（先写个位），小模型更容易学。"""
    return f"{a:02d}*{b:02d}=" + str(a * b)[::-1].ljust(A_LEN, "0") + ";"

In [ ]:
def make_sets(seed=0):
    rng = random.Random(seed)
    problems = [(a, b) for a in range(10, 100) for b in range(10, 100)]
    rng.shuffle(problems)
    return problems[:200], problems[200:400], problems[400:]  # benchmark, fresh, the rest / 基准、新题、其余

In [ ]:
def encode(problems, rewrite=False):
    """rewrite=True writes the problem the other way round (58*37 instead of 37*58): the same fact,
    different characters. rewrite=True 把题目反过来写：同一个事实，不同的字符。"""
    return torch.tensor([[STOI[c] for c in (fmt(b, a) if rewrite else fmt(a, b))] for a, b in problems])

In [ ]:
def train(train_problems, leaked=(), reps=0, rewrite=False, steps=1500, seed=0):
    torch.manual_seed(seed)
    data = encode(train_problems)
    if reps:
        data = torch.cat([data] + [encode(leaked, rewrite)] * reps)
    model = GPT(len(CHARS), d=64, n_layers=2, n_heads=4, ctx=Q_LEN + A_LEN + 1)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.1)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        x = data[torch.randint(0, len(data), (64,), generator=g)]
        logits = model(x[:, :-1])[:, Q_LEN - 1:]  # predict only the answer / 只预测答案
        loss = F.cross_entropy(logits.flatten(0, 1), x[:, Q_LEN:].flatten())
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model.eval()

In [ ]:
@torch.no_grad()
def score(model, problems):
    """Per-problem: correct? and the loss (nats) of the true answer. 每道题：对不对，以及正确答案的损失。"""
    x = encode(problems)
    logits = model(x[:, :-1])[:, Q_LEN - 1:Q_LEN - 1 + A_LEN]
    target = x[:, Q_LEN:Q_LEN + A_LEN]
    correct = (logits.argmax(-1) == target).all(-1)
    loss = F.cross_entropy(logits.transpose(1, 2), target, reduction="none").sum(-1)
    return correct.float(), loss

In [ ]:
def auc(pos, neg):
    """Probability that a random benchmark item has lower loss than a random fresh item (0.5 = no signal).
    随机一道基准题的损失低于随机一道新题的概率（0.5 = 看不出区别）。"""
    pos, neg = pos.tolist(), neg.tolist()
    return sum((p < n) + 0.5 * (p == n) for p in pos for n in neg) / (len(pos) * len(neg))

In [ ]:
def exact_overlap(train_strings, benchmark, n=9):
    """The usual decontamination check: does any n-character span of a benchmark item appear in the
    training data? (GPT-3 used 13-gram overlap on words.) 常用的去污染检查：基准题的任意 n 字符片段是否出现在训练数据里。"""
    grams = {s[i:i + n] for s in train_strings for i in range(len(s) - n + 1)}
    return sum(any(fmt(a, b)[i:i + n] in grams for i in range(len(fmt(a, b)) - n + 1)) for a, b in benchmark) / len(benchmark)

In [ ]:
def main():
    bench, fresh, rest = make_sets()
    train_set = rest[:3000]
    print("== 1. the benchmark / 基准 ==")
    print(f"  {len(bench)} problems such as {fmt(*bench[0])!r} (answer reversed); {len(fresh)} fresh twins; {len(train_set)} training problems")

    print("\n== 2. contamination inflates the benchmark, not the skill / 污染抬高的是基准分数，不是能力 ==")
    rows = {}
    for reps in [0, 1, 4]:
        model = train(train_set, bench, reps)
        (cb, lb), (cf, lf) = score(model, bench), score(model, fresh)
        rows[reps] = (cb.mean().item(), cf.mean().item(), auc(lb, lf))
        print(f"  leaked {reps}x: benchmark {rows[reps][0]:5.1%} | fresh {rows[reps][1]:5.1%} | gap {rows[reps][0] - rows[reps][1]:+5.1%} "
              f"| loss-based detection AUC {rows[reps][2]:.2f}")
    assert abs(rows[0][0] - rows[0][1]) < 0.08            # clean: the two sets agree / 干净：两组一致
    assert rows[4][0] > rows[0][0] + 0.25                 # contaminated: benchmark score jumps / 污染：基准分数大涨
    assert abs(rows[4][1] - rows[0][1]) < 0.10            # ... but fresh problems don't / 新题不变
    assert rows[4][2] > 0.7 > 0.6 > rows[0][2]            # the loss gap gives it away / 损失差暴露了污染

    print("\n== 3. detection by searching the training data / 在训练数据里搜索 ==")
    leak = [fmt(a, b) for a, b in bench]
    rewritten = [fmt(b, a) for a, b in bench]
    clean_strings = [fmt(a, b) for a, b in train_set]
    print(f"  9-character overlap with the training data: clean {exact_overlap(clean_strings, bench):.0%}, "
          f"verbatim leak {exact_overlap(clean_strings + leak, bench):.0%}, rewritten leak (58*37 for 37*58) {exact_overlap(clean_strings + rewritten, bench):.0%}")
    model = train(train_set, bench, 4, rewrite=True)
    (cb, lb), (cf, lf) = score(model, bench), score(model, fresh)
    print(f"  the rewritten leak still helps: benchmark {cb.mean():.1%} vs fresh {cf.mean():.1%}; loss AUC {auc(lb, lf):.2f}")
    assert exact_overlap(clean_strings, bench) < 0.02 and exact_overlap(clean_strings + rewritten, bench) < 0.1 < 0.95 < exact_overlap(clean_strings + leak, bench)
    assert cb.mean() > cf.mean() + 0.1
    print("  exact matching misses rewritten leaks; comparing against fresh items from the same distribution does not")

### 运行演示 / Run the demo (`contamination.py`)

In [ ]:
main()

## `llm_judge.py`

```text
LLM-as-a-judge and arena ratings / 用大模型当评委，以及竞技场评分.

Run / 运行:
    python docs/23-evaluation/code/llm_judge.py

Covers / 包含:
    1. a pairwise judge with the biases measured in real LLM judges: it prefers the first answer
       (position bias) and longer answers (verbosity bias), on top of ordinary noise
       一个两两比较的评委，带有真实大模型评委身上测到的偏差：偏爱第一个答案（位置偏差）和更长的答案（冗长偏差），再加上普通的噪声
    2. fixes: judge both orders and keep only consistent verdicts; fit and remove the length effect
       (as length-controlled AlpacaEval does)
       修正：两种顺序各判一次、只保留一致的结论；拟合并去掉长度的影响（如 AlpacaEval 的长度控制）
    3. arena ratings from pairwise votes: online Elo depends on the order of the games; a Bradley-Terry
       fit does not; bootstrap confidence intervals
       从两两投票得到竞技场评分：在线 Elo 依赖比赛顺序，Bradley-Terry 拟合不依赖；自助法置信区间

The judge is simulated with known biases, so its errors can be measured exactly. A few seconds.
评委是带已知偏差的模拟，所以它的错误可以被精确测量。几秒钟。
```

In [ ]:
__file__ = str(CODE / "llm_judge.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import numpy as np

In [ ]:
RNG = np.random.default_rng(0)

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

### 1. A biased pairwise judge / 有偏差的两两评委

In [ ]:
def make_answers(n, rng=RNG):
    """True quality (what a careful expert would judge) and length in words, independent of each other.
    真实质量（细心的专家会给出的判断）和长度（词数），两者互相独立。"""
    return rng.normal(0, 1, n), rng.lognormal(np.log(150), 0.5, n)

In [ ]:
class Judge:
    """Says whether the FIRST answer shown is better. perceived = quality + verbosity * log(length) + noise,
    and the first answer gets a bonus. 判断“先给出的答案”是否更好；感知分 = 质量 + 冗长系数 × log(长度) + 噪声，第一个答案额外加分。"""

    def __init__(self, position=0.35, verbosity=0.6, noise=0.5, rng=RNG):
        self.position, self.verbosity, self.noise, self.rng = position, verbosity, noise, rng

    def first_wins(self, q1, l1, q2, l2):
        s1 = q1 + self.verbosity * np.log(l1) + self.position + self.noise * self.rng.normal(size=np.shape(q1))
        s2 = q2 + self.verbosity * np.log(l2) + self.noise * self.rng.normal(size=np.shape(q2))
        return s1 > s2

In [ ]:
def evaluate_judge(judge, n=4000, rng=RNG):
    qa, la = make_answers(n, rng)
    qb, lb = make_answers(n, rng)
    truth = qa > qb
    ab = judge.first_wins(qa, la, qb, lb)          # A shown first / A 在前
    ba = ~judge.first_wins(qb, lb, qa, la)         # B shown first; True means A wins / B 在前；True 表示 A 胜
    consistent = ab == ba
    return {"single order": (ab == truth).mean(), "consistent": consistent.mean(),
            "consistent only": (ab[consistent] == truth[consistent]).mean(),
            "first wins, equal answers": judge.first_wins(qa, la, qa, la).mean()}

In [ ]:
def padded_win_rate(judge, factor=2.0, n=4000, rng=RNG):
    """The same answer, once as is and once padded to `factor` times the length with no new content;
    shown in random order. 同一个答案，一次原样、一次加长到 factor 倍但不增加内容；随机先后。"""
    q, l = make_answers(n, rng)
    first = rng.random(n) < 0.5
    w = judge.first_wins(q, np.where(first, l * factor, l), q, np.where(first, l, l * factor))
    return np.where(first, w, ~w).mean()

In [ ]:
def fit_logistic(X, y, steps=4000, lr=0.5):
    w = np.zeros(X.shape[1])
    for _ in range(steps):  # logistic regression by gradient descent / 梯度下降做逻辑回归
        w -= lr * X.T @ (sigmoid(X @ w) - y) / len(y)
    return w

In [ ]:
def length_controlled_win_rate(judge, factor=2.0, n=4000, rng=RNG):
    """The padded "model" vs the original, as length-controlled AlpacaEval does it: fit
    P(A wins) = sigmoid(theta * [A is padded - B is padded] + c * log(len A / len B)) on these comparisons
    together with ordinary comparisons of varied length (where only length differs systematically), then
    report sigmoid(theta): the win rate with the length difference set to zero.
    像长度控制的 AlpacaEval 一样：把加长版与原版的比较，和长度各异的普通比较放在一起拟合，再报告长度差为零时的胜率。"""
    rows, ys = [], []
    q, l = make_answers(n, rng)                    # padded vs original / 加长版对原版
    for padded_first in (True, False):
        l1, l2 = (l * factor, l) if padded_first else (l, l * factor)
        rows.append(np.column_stack([np.full(n, 1.0 if padded_first else -1.0), np.log(l1 / l2)]))
        ys.append(judge.first_wins(q, l1, q, l2))
    qa, la = make_answers(n, rng)                  # ordinary comparisons / 普通比较
    qb, lb = make_answers(n, rng)
    for first in ((qa, la, qb, lb), (qb, lb, qa, la)):
        rows.append(np.column_stack([np.zeros(n), np.log(first[1] / first[3])]))
        ys.append(judge.first_wins(*first))
    X, y = np.vstack(rows), np.concatenate(ys).astype(float)
    theta, c = fit_logistic(X, y)
    return sigmoid(theta), c

### 3. Arena ratings / 竞技场评分

In [ ]:
def battles(strengths, n, rng=RNG):
    m = len(strengths)
    a = rng.integers(0, m, n)
    b = (a + rng.integers(1, m, n)) % m  # a different opponent / 不同的对手
    a_wins = rng.random(n) < sigmoid(strengths[a] - strengths[b])
    return a, b, a_wins

In [ ]:
def elo(a, b, a_wins, m, k=32):
    """Online Elo, as in chess: update after every game, in the order they were played. 在线 Elo：按比赛顺序逐场更新。"""
    r = np.full(m, 1000.0)
    for i, j, w in zip(a, b, a_wins):
        e = 1 / (1 + 10 ** ((r[j] - r[i]) / 400))
        r[i] += k * (w - e)
        r[j] -= k * (w - e)
    return r

In [ ]:
def bradley_terry(a, b, a_wins, m, iters=200):
    """Maximum-likelihood strengths for P(i beats j) = p_i / (p_i + p_j), by the MM algorithm; returned on
    the Elo scale (400 log10). Order of games does not matter. 用 MM 算法求 Bradley-Terry 的极大似然强度，与比赛顺序无关。"""
    wins = np.zeros(m)
    np.add.at(wins, a, a_wins)
    np.add.at(wins, b, ~a_wins)
    games = np.zeros((m, m))
    np.add.at(games, (a, b), 1)
    games += games.T
    p = np.ones(m)
    for _ in range(iters):
        p = wins / (games / (p[:, None] + p[None, :])).sum(1)
        p /= np.exp(np.log(p).mean())
    return 400 * np.log10(p) + 1000

In [ ]:
def main():
    print("== 1. a biased judge / 有偏差的评委 ==")
    judge = Judge()
    r = evaluate_judge(judge)
    print(f"  agrees with the expert {r['single order']:.0%} when each pair is judged once")
    print(f"  identical answers in both slots: the first one wins {r['first wins, equal answers']:.0%} (an unbiased judge: 50%)")
    print(f"  same verdict after swapping the order: {r['consistent']:.0%} of pairs")
    pad = padded_win_rate(judge)
    print(f"  the same answer padded to twice the length, with nothing new: wins {pad:.0%} against itself")
    assert r["first wins, equal answers"] > 0.6 and pad > 0.6 and r["consistent"] < 0.9

    print("\n== 2. fixes / 修正 ==")
    print(f"  judge both orders, keep consistent verdicts: {r['consistent only']:.0%} agreement on the {r['consistent']:.0%} kept (rest = tie)")
    lc, c = length_controlled_win_rate(judge)
    print(f"  length-controlled win rate of the padded answers: {lc:.0%} (raw {pad:.0%}); fitted length effect {c:.2f} per log-unit")
    assert r["consistent only"] > r["single order"] + 0.05
    assert 0.5 < lc < pad - 0.05
    print("  it removes part of the bias, not all: a simple statistical model of the judge only approximates it")
    unbiased = evaluate_judge(Judge(position=0, verbosity=0))
    print(f"  for reference, a judge with only noise: {unbiased['single order']:.0%} agreement, {unbiased['consistent']:.0%} consistent")

    print("\n== 3. arena ratings / 竞技场评分 ==")
    strengths = np.linspace(-1.2, 1.2, 8)
    a, b, w = battles(strengths, 3000)
    bt = bradley_terry(a, b, w, 8)
    elos = []
    for _ in range(20):
        order = RNG.permutation(len(a))
        elos.append(elo(a[order], b[order], w[order], 8))
    elos = np.array(elos)
    print("  true  " + " ".join(f"{400 * s / np.log(10) + 1000:6.0f}" for s in strengths))
    print("  BT    " + " ".join(f"{v:6.0f}" for v in bt))
    print("  Elo   " + " ".join(f"{v:6.0f}" for v in elos.mean(0)) + "  (mean of 20 game orders)")
    spread = elos.std(0).mean()
    print(f"  the same 3000 votes in 20 different orders: Elo ratings move by ±{spread:.0f} on average; Bradley-Terry by 0")
    boots = []
    for _ in range(200):
        idx = RNG.integers(0, len(a), len(a))
        boots.append(bradley_terry(a[idx], b[idx], w[idx], 8))
    lo, hi = np.percentile(np.array(boots), [2.5, 97.5], axis=0)
    print("  BT 95% intervals: " + " ".join(f"[{l:.0f},{h:.0f}]" for l, h in zip(lo, hi)))
    true_elo = 400 * strengths / np.log(10) + 1000
    assert np.all(np.diff(bt) > 0) and np.abs(bt - true_elo).max() < 60
    assert spread > 5
    assert np.mean((lo <= true_elo) & (true_elo <= hi)) >= 0.75

### 运行演示 / Run the demo (`llm_judge.py`)

In [ ]:
main()

## `perplexity.py`

```text
Perplexity: what it measures and when it can be compared / 困惑度：衡量什么，什么时候能比较.

Run / 运行:
    python docs/23-evaluation/code/perplexity.py

Covers / 包含:
    1. perplexity = exp(average negative log-likelihood per token), and its floor: the entropy of the
       text itself (computed exactly here, because chapter 6's toy language comes from a known generator)
       困惑度 = exp(每个词元的平均负对数似然)，以及它的下限：文本本身的熵（这里能精确算出，因为 06 章的小语言来自已知的生成器）
    2. the same text, two tokenizers: a word-level and a character-level model have very different
       perplexities; bits per character makes them comparable
       同一段文本、两种分词：词级和字符级模型的困惑度相差很大；换算成每字符比特数才能比较
    3. a cheap trick: merging words into one <unk> class lowers perplexity although the model knows less
       一个“作弊”：把一些词合并成 <unk>，困惑度下降，模型却知道得更少

About a minute on a laptop CPU. 笔记本 CPU 上约一分钟。
```

In [ ]:
__file__ = str(CODE / "perplexity.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import math
import random
import sys
from pathlib import Path
import torch
import torch.nn.functional as F

In [ ]:
sys.path.insert(0, str(Path(__file__).resolve().parent.parent.parent / "06-pretraining" / "code"))

In [ ]:
import pretrain as P  # noqa: E402

In [ ]:
LN2 = math.log(2)

### The exact probability of a sentence under the generator / 一个句子在生成器下的精确概率

In [ ]:
def np_logprob(words, i, nouns):
    """log P of the noun phrase starting at words[i], and where it ends. 从 i 开始的名词短语的对数概率及结束位置。"""
    det, lp = words[i], 0.0
    plural_forms = set(nouns.values())
    j = i + 1
    adj = words[j] in P.ADJECTIVES
    noun = words[j + 1] if adj else words[j]
    plural = noun in plural_forms
    lp += math.log(1 / 4 if plural else 1 / 3)  # determiner / 限定词 ("an" is a spelling of "a")
    lp += math.log(0.4 / len(P.ADJECTIVES)) if adj else math.log(0.6)
    lp += math.log(1 / len(nouns))
    return lp, j + (2 if adj else 1), plural

In [ ]:
def true_logprob(words):
    """log P(sentence) under chapter 6's generator (natural log), following sentence() step by step.
    按 06 章的 sentence() 逐步计算句子的对数概率。"""
    lp, i, plural = 0.0, 0, None
    lp_np, i, plural = np_logprob(words, 0, P.SUBJECTS)
    lp += lp_np + math.log(0.5)  # subject number / 主语单复数
    if words[i] == "that":
        lp += math.log(0.3) + math.log(1 / len(P.TRANSITIVE))
        lp_np, i, obj_plural = np_logprob(words, i + 2, P.OBJECTS)
        lp += lp_np + math.log(0.5)
    else:
        lp += math.log(0.7)
    v = words[i]
    if v in P.INTRANSITIVE or v in P.INTRANSITIVE.values():
        lp += math.log(0.5) + math.log(1 / len(P.INTRANSITIVE))
        i += 1
    else:
        lp += math.log(0.5) + math.log(1 / len(P.TRANSITIVE))
        lp_np, i, _ = np_logprob(words, i + 1, P.OBJECTS)
        lp += lp_np + math.log(0.5)
    if words[i] in P.PREPOSITIONS:
        lp += math.log(0.4) + math.log(1 / len(P.PREPOSITIONS)) + math.log(1 / len(P.PLACES))
        i += 3
    else:
        lp += math.log(0.6)
    if words[i] in P.ADVERBS:
        lp += math.log(0.3) + math.log(1 / len(P.ADVERBS))
        i += 1
    else:
        lp += math.log(0.7)
    assert words[i] == "." and i == len(words) - 1, words
    return lp

In [ ]:
def sentences(n, seed):
    rng = random.Random(seed)
    return [P.sentence(rng) for _ in range(n)]

### Two models of the same text / 同一段文本的两个模型

In [ ]:
CHARS = sorted({c for s in sentences(3000, 1) for c in " ".join(s) + " "})

In [ ]:
def word_ids(sents):
    return [P.STOI[w] for s in sents for w in s]

In [ ]:
def char_ids(sents):
    return [CHARS.index(c) for s in sents for c in " ".join(s) + " "]

In [ ]:
def train(ids, vocab, ctx, steps=700, seed=0):
    torch.manual_seed(seed)
    data = torch.tensor(ids)
    model = P.GPT(vocab, ctx=ctx)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, betas=(0.9, 0.95), weight_decay=0.1)
    g = torch.Generator().manual_seed(seed)
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = P.cosine_lr(step, steps, 3e-3, 50)
        s = torch.randint(0, len(data) - ctx - 1, (32,), generator=g)
        x = torch.stack([data[i:i + ctx] for i in s])
        y = torch.stack([data[i + 1:i + ctx + 1] for i in s])
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model.eval()

In [ ]:
@torch.no_grad()
def token_nll(model, ids, ctx):
    """Negative log-likelihood (nats) of every token after the first, with a sliding window.
    用滑动窗口计算每个词元（第一个之外）的负对数似然。"""
    data = torch.tensor(ids)
    out = torch.zeros(len(ids) - 1)
    stride = ctx // 2
    for start in range(0, len(ids) - 1, stride):
        lo = max(0, start + stride - ctx)
        x = data[lo:start + stride][None]
        y = data[lo + 1:start + stride + 1]
        nll = F.cross_entropy(model(x)[0, :len(y)], y, reduction="none")
        keep = slice(start - lo, start - lo + min(stride, len(y) - (start - lo)))
        out[start:start + nll[keep].numel()] = nll[keep]
    return out

In [ ]:
def main():
    train_s, test_s = sentences(6000, 1), sentences(400, 2)
    text = " ".join(" ".join(s) for s in test_s)
    n_words, n_chars = sum(len(s) for s in test_s), len(text)

    print("== 1. perplexity and its floor / 困惑度与它的下限 ==")
    true_nats = -sum(true_logprob(s) for s in test_s)
    print(f"  the generator's own entropy: {true_nats / n_words:.3f} nats per word -> the lowest possible perplexity is {math.exp(true_nats / n_words):.2f}")
    word_model = train(word_ids(train_s), len(P.VOCAB), ctx=32)
    w_nll = token_nll(word_model, word_ids(test_s), 32)
    w_ppl = math.exp(w_nll.mean())
    print(f"  word-level GPT: {w_nll.mean():.3f} nats per word, perplexity {w_ppl:.2f} "
          f"(a uniform guess over {len(P.VOCAB)} words would be {len(P.VOCAB)})")
    assert math.exp(true_nats / n_words) < w_ppl < 1.3 * math.exp(true_nats / n_words)
    by_word = {}
    for v, i in zip(w_nll.tolist(), word_ids(test_s)[1:]):
        by_word.setdefault(P.VOCAB[i], []).append(v)
    avg = sorted(((sum(v) / len(v), w) for w, v in by_word.items()), reverse=True)
    print("  hardest words (avg nats): " + ", ".join(f"{w!r} {v:.1f}" for v, w in avg[:4]) +
          "; easiest: " + ", ".join(f"{w!r} {v:.2f}" for v, w in avg[-3:]))
    print("  the hard ones are genuinely random choices of the generator, not model mistakes")

    print("\n== 2. the same text, two tokenizers / 同一段文本，两种分词 ==")
    char_model = train(char_ids(train_s), len(CHARS), ctx=128)
    c_nll = token_nll(char_model, char_ids(test_s), 128)
    c_ppl = math.exp(c_nll.mean())
    w_bpc = w_nll.sum().item() / LN2 / n_chars
    c_bpc = c_nll.sum().item() / LN2 / n_chars
    true_bpc = true_nats / LN2 / n_chars
    print(f"  perplexity: word-level {w_ppl:.2f} per word vs character-level {c_ppl:.2f} per character (not comparable)")
    print(f"  bits per character: word-level {w_bpc:.3f}, character-level {c_bpc:.3f}, generator {true_bpc:.3f}")
    assert w_ppl > 3 * c_ppl                       # perplexities look wildly different / 困惑度看起来差很多
    assert true_bpc < w_bpc < 2 * true_bpc and true_bpc < c_bpc < 2 * true_bpc  # same scale, both above the floor / 同一量级，都在下限之上
    print(f"  by perplexity the word-level model looked {w_ppl / c_ppl:.0f}x worse; per character it is the better one")

    print("\n== 3. merging words lowers perplexity / 合并词语会降低困惑度 ==")
    with torch.no_grad():
        merged = set(P.STOI[w] for w in list(P.OBJECTS.values()) + list(P.OBJECTS) + P.PLACES)
        ids = word_ids(test_s)
        x = torch.tensor(ids)
        nll = []
        for i in range(0, len(ids) - 32, 32):
            logp = F.log_softmax(word_model(x[i:i + 32][None])[0], -1)
            for t, y in enumerate(ids[i + 1:i + 33]):
                if y in merged:  # P(<unk>) = sum of the merged words' probabilities / 合并类的概率 = 各词概率之和
                    nll.append(-torch.logsumexp(logp[t, list(merged)], 0).item())
                else:
                    nll.append(-logp[t, y].item())
    unk_ppl = math.exp(sum(nll) / len(nll))
    print(f"  {len(merged)} object and place words merged into <unk>: perplexity {w_ppl:.2f} -> {unk_ppl:.2f}, "
          "although the model now says nothing about which object or place")
    assert unk_ppl < w_ppl
    print("  compare perplexities only on the same text with the same tokenizer and vocabulary; otherwise use bits per character or byte")

### 运行演示 / Run the demo (`perplexity.py`)

In [ ]:
main()

## 动手试试 / Try it yourself

改一改上面任意一个函数或参数，重新运行它所在的单元格，再运行 `main()` 看结果有什么变化。
assert 失败说明你的改动改变了程序检查的性质，这正是值得思考的地方。

Change any function or parameter above, re-run its cell, then run `main()` again. If an assert fails, your change broke a
property the script checks, which is exactly what is worth thinking about.